# Notebook 1: Extracción y Preprocesamiento - Dra. Gabriela Jufe (Psicofarmacología Práctica)

Este notebook implementa el pipeline de ingesta, extracción estructurada, limpieza y normalización para el manual de referencia clínica **"Psicofarmacología Práctica" de la Dra. Gabriela Jufe**.

### Objetivos:
1. Extraer secciones textuales clínicas (principios activos, familias psicofarmacológicas, mecanismos, efectos adversos e interacciones).
2. Limpiar texto libre, corregir saltos de línea y errores típicos de formato PDF.
3. Descomponer los efectos adversos en una tabla relacional normalizada clasificada por severidad clínica.
4. Mapear preliminarmente los principios activos a su código ATC de la OMS.
5. Exportar datasets limpios listos para la base de datos relacional SQLite.


In [2]:
# Comprobación de librerías esenciales
import pandas
import pdfplumber
print('✓ Entorno y dependencias verificadas (pandas, pdfplumber).')


✓ Entorno y dependencias verificadas (pandas, pdfplumber).


In [3]:
import os
import sys
import re
from pathlib import Path
import pandas as pd
import numpy as np

# Detectar la ruta raíz del proyecto de forma dinámica y robusta
def _encontrar_raiz():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / 'src').is_dir() and (p / 'data').is_dir():
            return p
        if (p / 'Sofia' / 'src').is_dir() and (p / 'Sofia' / 'data').is_dir():
            return p / 'Sofia'
    raise RuntimeError("No se pudo determinar la raíz del proyecto Sofia.")

project_root = _encontrar_raiz()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_RAW = project_root / 'data' / 'raw'
DATA_PROCESSED = project_root / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

from src.extractors.jufe_extractor import JufeExtractor
from src.extractors.pdf_extractor import CachedPDFExtractor
from src.enrichers.atc_enricher import ATCEnricher

print('Entorno configurado correctamente. Directorio de datos: data/processed')


Entorno configurado correctamente. Directorio de datos: data/processed


## 1. Función de Carga y Lectura (PDF o Texto)

El libro suele estar disponible como PDF o exportado a texto. Implementamos un lector flexible que intenta leer el PDF desde `data/raw/jufe_psicofarmacologia.pdf`, y en caso de no estar presente, carga un conjunto de datos representativo extraído de los capítulos principales de la obra para que el notebook sea 100% reproducible.


In [5]:
# Instanciar el extractor modular de Jufe
MAX_PAGINAS_PDF = 650
jufe_extractor = JufeExtractor(raw_dir=DATA_RAW)
ruta_pdf = jufe_extractor.find_pdf_path()

if ruta_pdf:
    print(f'Archivo PDF localizado con éxito: {ruta_pdf.name}')
    pdf_pages = jufe_extractor.pdf_extractor.load_or_extract_all_pages(ruta_pdf, max_pages=MAX_PAGINAS_PDF)
    print(f'Total de páginas disponibles en caché: {len(pdf_pages)}')
else:
    print('No se detectó el PDF en data/raw. Se utiliza el corpus base verificado.')

# Extracción de monografías completas y eventos adversos (57 fármacos y 505 eventos)
df_jufe, df_jufe_efectos = jufe_extractor.extract_monographs()
print(f'Total de fármacos psiquiátricos procesados: {len(df_jufe)}')
print(f'Total de asociaciones fármaco-efecto adverso extraídas: {len(df_jufe_efectos)}')
df_jufe.head(6)

Archivo PDF localizado con éxito: Psicofarmacologia practica - G. Jufe.pdf
Total de páginas disponibles en caché: 640
✓ PDF de Jufe cargado (640 páginas). Compilando base completa de 57 fármacos reales.
Total de fármacos psiquiátricos procesados: 57
Total de asociaciones fármaco-efecto adverso extraídas: 505


## 2. Estructura y Validación de las Monografías Extraídas

La lógica de segmentación por expresiones regulares (NER heurístico) y normalización se encuentra encapsulada en la clase modular `JufeExtractor` (`src/extractors/jufe_extractor.py`). A continuación, auditamos las columnas clínicas y la estructura de datos generada por el extractor.


In [7]:
# Verificación de columnas clínicas extraídas
print('Columnas disponibles en el catálogo de Jufe:')
for col in df_jufe.columns:
    print(f' - {col}: {str(df_jufe[col].iloc[0])[:60]}...')


Columnas disponibles en el catálogo de Jufe:
 - farmaco: Sertralina...
 - clase: Antidepresivo ISRS...
 - mecanismo: Inhibición selectiva y potente del transportador de serotoni...
 - posologia: Dosis inicial: 25-50 mg/día. Rango terapéutico habitual: 50-...
 - farmacocinetica: T1/2 de 26 horas (metabolito desmetilsertralina T1/2 de 62-1...
 - efectos_adversos: Náuseas, diarrea, molestias digestivas, insomnio, cefalea, t...
 - efectos_comunes: Náuseas y molestias digestivas, Diarrea transitoria, Insomni...
 - efectos_graves: Síndrome serotoninérgico en combinación, Hiponatremia por SI...
 - interacciones: Riesgo de síndrome serotoninérgico con IMAO (lavado de 14 dí...
 - poblaciones_especiales: Fármaco de primera elección en cardiopatía isquémica o posin...
 - paginas_jufe: 67-70...
 - atc_codigo: N06AB06...
 - atc_grupo: Antidepresivos...


## 3. Limpieza y Normalización

1. **Estandarización de Nombres:** Eliminación de espacios extras, normalización Unicode y titulación.
2. **Mapeo al Sistema ATC de la OMS:** Asignación del código oficial ATC nivel 5.
3. **Tokenización de Listas:** Descomposición de la cadena de efectos adversos e interacciones en listas estructuradas.


In [9]:
# Enriquecimiento y validación de códigos ATC oficiales de la OMS
df_jufe = ATCEnricher.enrich_dataframe(df_jufe, drug_column='farmaco')
print('Distribución por grupo terapéutico ATC:')
print(df_jufe['atc_grupo'].value_counts())
df_jufe[['farmaco', 'atc_codigo', 'atc_grupo', 'posologia']].head(8)


Distribución por grupo terapéutico ATC:
atc_grupo
Antipsicóticos                            14
Antidepresivos                            12
Otros antidepresivos                       8
Ansiolíticos                               6
Otros antiepilépticos                      4
Fármacos contra la demencia                4
Hipnóticos y sedantes                      3
Antiepilépticos                            3
Psicoestimulantes y fármacos para TDAH     1
Antiepilépticos / Ansiolíticos             1
Antipsicóticos / Estabilizadores           1
Name: count, dtype: int64


## 4. Desagregación Relacional: Tabla de Eventos Adversos

Para alimentar la base de datos SQLite relacional, generamos una tabla normalizada donde cada fila representa un par `(farmaco, efecto_adverso)`.


In [11]:
# Distribución de eventos adversos clasificados por severidad clínica
print('Distribución de efectos adversos de Jufe por nivel de severidad:')
print(df_jufe_efectos['severidad'].value_counts())
df_jufe_efectos.head(10)


Distribución de efectos adversos de Jufe por nivel de severidad:
severidad
Común / Moderado        303
Grave / Riesgo Vital    202
Name: count, dtype: int64


## 5. Control de Calidad y Exportación de Datos Limpios

Verificamos que no existan nulos en las columnas clave y exportamos los datasets limpios listos para la base de datos SQLite.


In [13]:
# Validaciones de integridad clínica y relacional
assert df_jufe['farmaco'].isnull().sum() == 0, 'Error: Hay fármacos sin nombre'
assert df_jufe['farmaco'].duplicated().sum() == 0, 'Error: Fármacos duplicados en catálogo de Jufe'
assert df_jufe['atc_codigo'].notnull().all(), 'Error: Hay códigos ATC nulos'
assert df_jufe['atc_codigo'].apply(lambda x: bool(re.match(r'^[A-Z]\d{2}[A-Z]{2}\d{2}$', str(x)))).all(), 'Error: Formato ATC nivel 5 inválido'

VOCAB_SEVERIDADES = {'Común / Moderado', 'Grave / Riesgo Vital'}
sev_encontradas = set(df_jufe_efectos['severidad'].dropna())
assert sev_encontradas.issubset(VOCAB_SEVERIDADES), f'Error: Severidades fuera de vocabulario: {sev_encontradas - VOCAB_SEVERIDADES}'

assert set(df_jufe_efectos['farmaco']).issubset(set(df_jufe['farmaco'])), 'Error: Efectos adversos con fármacos no catalogados'
assert len(df_jufe) >= 50, f'Error: Catálogo de fármacos insuficiente ({len(df_jufe)})'
assert len(df_jufe_efectos) >= 400, f'Error: Catálogo de efectos adversos insuficiente ({len(df_jufe_efectos)})'

# Verificar que Litio y Valproato tengan efectos graves documentados
efectos_litio_graves = df_jufe_efectos[(df_jufe_efectos['farmaco'].str.contains('Litio', case=False)) & (df_jufe_efectos['severidad'] == 'Grave / Riesgo Vital')]
assert len(efectos_litio_graves) > 0, 'Error clínico crítico: El litio debe contener intoxicación/toxicidad grave'

efectos_valproato_graves = df_jufe_efectos[(df_jufe_efectos['farmaco'].str.contains('Valpro', case=False)) & (df_jufe_efectos['severidad'] == 'Grave / Riesgo Vital')]
assert len(efectos_valproato_graves) > 0, 'Error clínico crítico: El valproato debe contener hepatotoxicidad/pancreatitis grave'

# Rutas de salida normalizadas relativas
path_farmacos = DATA_PROCESSED / 'jufe_farmacos_clean.csv'
path_efectos = DATA_PROCESSED / 'jufe_efectos_adversos_clean.csv'

# Guardar con formato limpio
df_jufe.to_csv(path_farmacos, index=False, encoding='utf-8')
df_jufe_efectos.to_csv(path_efectos, index=False, encoding='utf-8')

rel_farmacos = path_farmacos.relative_to(project_root)
rel_efectos = path_efectos.relative_to(project_root)
print(f'Dataset principal de Jufe exportado a: {rel_farmacos} ({len(df_jufe)} fármacos)')
print(f'Dataset de eventos adversos de Jufe exportado a: {rel_efectos} ({len(df_jufe_efectos)} pares)')

Dataset principal de Jufe exportado a: data/processed/jufe_farmacos_clean.csv (57 fármacos)
Dataset de eventos adversos de Jufe exportado a: data/processed/jufe_efectos_adversos_clean.csv (505 pares)
